In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax

df = pd.read_csv('./datasets/employee_reviews_indeed_latest_cleaned.csv')

c:\Users\dougl\Documents\School\TXSA\Assignment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print(f"Before dropping NAs: {df.shape}")
df = df.dropna()
print(f"After dropping NAs: {df.shape}")

Before dropping NAs: (7024, 4)
After dropping NAs: (7024, 4)


,company,review_date,rating,clean_text
0,Shell,2025-04-21,5.0,team work / challenges transparency / fast pac...
1,Shell,2025-04-22,2.0,not like it used to be. 30 days holiday. bad m...
2,Shell,2025-04-24,1.0,long shifts. shifts were long with rubbish pay...
3,Shell,2025-04-25,5.0,really enjoyed working with the team and leade...
4,Shell,2025-04-26,5.0,educational. what is the best part of working ...
...,...,...,...,...
7019,Spotify,2025-07-18,5.0,fun and rewarding. great place to work. greak ...
7020,Spotify,2025-07-31,5.0,good culture. spotify s culture is rooted in i...
7021,Spotify,2025-08-18,4.0,fun and creative work opportunities. i love wo...
7022,Spotify,2025-08-21,4.0,i want to earn on spotify. a typical day at wo...


In [5]:
# For rating labelling
df_2 = df.copy()

In [ ]:
# Using sentimental model for labelling
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

def model_predict(text):
    encoded_input = tokenizer(text, return_tensors='pt')
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)

    ranking = np.argsort(scores)
    ranking = ranking[::-1]

    return config.id2label[ranking[0]]

df['sentiment'] = df['clean_text'].apply(model_predict)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22811.78it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,company,review_date,rating,clean_text,sentiment
0,Shell,2025-04-21,5.0,team work / challenges transparency / fast pac...,positive
1,Shell,2025-04-22,2.0,not like it used to be. 30 days holiday. bad m...,negative
2,Shell,2025-04-24,1.0,long shifts. shifts were long with rubbish pay...,negative
3,Shell,2025-04-25,5.0,really enjoyed working with the team and leade...,positive
4,Shell,2025-04-26,5.0,educational. what is the best part of working ...,positive
...,...,...,...,...,...
7019,Spotify,2025-07-18,5.0,fun and rewarding. great place to work. greak ...,positive
7020,Spotify,2025-07-31,5.0,good culture. spotify s culture is rooted in i...,positive
7021,Spotify,2025-08-18,4.0,fun and creative work opportunities. i love wo...,positive
7022,Spotify,2025-08-21,4.0,i want to earn on spotify. a typical day at wo...,positive


In [ ]:
print(df['sentiment'].value_counts())
print(f"\nBefore dropping neutral sentiments: {df.shape}")
df.to_csv('./datasets/employee_reviews_indeed_latest_cleaned_labeled_with_sentiment_model.csv', index=False)

sentiment
positive    4888
negative    1693
neutral      443
Name: count, dtype: int64

Before dropping neutral sentiments: (7024, 5)


In [ ]:
# Rating Labelling
def label_rating(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'
    
df2['sentiment'] = df2['rating'].apply(label_rating)
print(df2['sentiment'].value_counts())
print(f"\nBefore dropping neutral sentiments: {df2.shape}")


sentiment
positive    4449
neutral     1386
negative    1189
Name: count, dtype: int64

Before dropping neutral sentiments: (7024, 5)


In [ ]:
print(df2['sentiment'].value_counts())
print(f"\nBefore dropping neutral sentiments: {df2.shape}")
df2.to_csv('./datasets/employee_reviews_indeed_latest_cleaned_labeled_with_rating.csv', index=False)